# Agentic workflows with local LLMs
## Hands-on companion notebook — AI for Urban Health (Summer Institute 2026), Day 3

This notebook walks through three increasingly capable patterns for building **agentic workflows** on a
**local** model running in Ollama — so it is free to run and your data never leaves your machine:

1. **One prompt** — a single structured-output call to extract fields from a public-health abstract.
2. **Multi-agent (CrewAI)** — a screening agent and an extraction agent collaborating on a list of abstracts.
3. **State machine (LangGraph)** — the same workflow as an explicit graph with conditional branching and per-step control.

Then a short **tool-using** example and a **try it yourself** exercise.

It maps onto the Day 3 lecture: the *agent loop*, *context engineering*, *tools / MCP*, *guardrails*, and
*verification as the bottleneck*. It is an **optional, advanced track** — a richer companion to the basic
Ollama activity. Your instructor may demo it live; you can also work through it on your own afterward.

### Choosing a model (mid-2026)

Open models move fast. As of mid-2026 the strongest *open-weight* model is **GLM-5.2** (Z.ai, MIT license, ~744B
mixture-of-experts) — but it is a **datacenter/cloud-class** model, not something you run on a laptop. For
**local, laptop-friendly** use, the best options are smaller members of current families:

- **Gemma 4** (Google) — small variants run on modest laptops.
- **Qwen3** (Alibaba) — 4B–14B sizes are strong at structured output and tool use; good default for this notebook.
- **Llama 4** small / **Phi-4** — solid lightweight alternatives.

See the laptop sizing table in the next cell. **Model names and tags change often — run `ollama list` and browse
<https://ollama.com/library> for the exact current tags and sizes.**

**Prerequisites**
- [Ollama](https://ollama.com) installed and running (the llama icon visible in your menu bar or system tray).
- A local model pulled. In a terminal, once (pick one that fits your laptop — see next cell):

```
ollama pull qwen3        # good default; try a smaller size like qwen3:4b on 8 GB laptops
ollama pull gemma4       # Google's current small/efficient family
```

Set the `MODEL` constant in the setup cell to whatever you pulled.

### Which model fits your laptop?

Rough guide using 4-bit quantization (`Q4_K_M`, Ollama's default). A model needs roughly **0.6 GB of memory per
billion parameters** at Q4, plus headroom for context.

| Your laptop | Usable memory | Models that run well (Q4) |
|---|---|---|
| Entry (8 GB RAM, no/integrated GPU) | ~5 GB for the model | Gemma 4 small, Qwen3 4B, Phi-4-mini, Llama 3.2 3B |
| Mainstream (16 GB RAM, or 8 GB GPU) | ~10 GB | Qwen3 8B, Gemma 4, Llama 3.1 8B |
| Strong (32 GB RAM, or 12–16 GB GPU) | ~20 GB | Qwen3 14B–32B, Gemma 4 (larger), gpt-oss 20B |
| Workstation (64 GB+ unified, or 24 GB GPU) | 40 GB+ | 70B-class or large MoE, heavily quantized |
| Not a laptop (cloud/server) | — | **GLM-5.2**, DeepSeek V4, MiniMax M3, Llama 4 Maverick — use Ollama Cloud or a hosted API |

**Mac (Apple Silicon — M1/M2/M3/M4):** memory is *unified* (shared by CPU and GPU), and Ollama uses the GPU
(Metal) automatically. Plan to use about 60–70% of total RAM for the model: a 16 GB Mac handles up to ~8–14B
comfortably, a 32 GB Mac up to ~32B, 64 GB+ for larger. *Intel Macs are CPU-only — stick to 3–4B models.*

**PC (Windows / Linux):** with an **NVIDIA GPU**, the limit is **VRAM** — the model should mostly fit in VRAM for
good speed (check with `nvidia-smi`). 8 GB VRAM ≈ up to 8B, 12 GB ≈ up to 14B, 24 GB (e.g., RTX 4090) ≈ up to ~32B.
**Without a dedicated GPU**, Ollama still runs on CPU + system RAM — fine for 3–8B models, just slower. AMD GPUs
work via ROCm on Linux (and recent Windows builds).

Quick check of what you already have: run `ollama list` (downloaded models) and, while a model is loaded,
`ollama ps` (shows whether it is running on GPU or CPU).

## 0. Setup

Install the Python packages we will use. This runs `pip` from inside the notebook; it is one-time per environment.

> In **Positron** or VS Code, select a Python interpreter first; in Jupyter/Colab this cell just works.

In [ ]:
# Run once. Safe to re-run; pip will skip already-installed packages.
%pip install --quiet ollama crewai "langgraph>=0.2" "langchain-ollama>=0.2" pydantic

Now confirm Ollama is running and the model is available.

In [ ]:
import ollama

MODEL = "qwen3"        # any local model you pulled; e.g. "qwen3:4b", "gemma4", "llama3.1:8b"

# Lists models currently downloaded on your machine.
available = [m.model for m in ollama.list().models]
print("Models available locally:", available)

assert any(MODEL in m for m in available), (
    f"{MODEL} not found. Run `ollama pull {MODEL}` in a terminal, "
    f"or set MODEL to one of: {available}"
)
print(f"\nUsing model: {MODEL}")

## 1. Hello, local model

A bare-bones single-prompt call. Useful for sanity-checking your install and for understanding what happens
under the hood when a framework calls the model.

In [ ]:
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a public health communications editor."},
        {"role": "user", "content": "In one short paragraph, explain what a confidence interval is to a community advisory board."}
    ]
)
print(response.message.content)

## 2. Sample data: urban-health abstracts

We will use six short, fictional but realistic abstracts to test the pipeline. They span pedestrian safety,
extreme heat, and air quality. Three should match our inclusion criteria (US, 2020 or later, an evaluation of a
built-environment / greening / traffic intervention, with an urban-health outcome); three should not.

In [ ]:
ABSTRACTS = [
    {
        "id": "A001",
        "text": (
            "Background: Vision Zero efforts in Philadelphia, PA installed leading pedestrian intervals (LPIs) "
            "at 117 signalized intersections from 2021 to 2023. Methods: We used a difference-in-differences "
            "design comparing pedestrian crash counts at treated and untreated intersections. Results: LPIs "
            "were associated with a 22% reduction in pedestrian-vehicle crashes (95% CI: 11%, 32%). Limitation: "
            "Crash reporting completeness varied across precincts."
        )
    },
    {
        "id": "A002",
        "text": (
            "Objective: To estimate the effect of statewide motorcycle helmet laws on motorcyclist fatalities "
            "in Texas, 2018-2024. Methods: Interrupted time series. Results: Helmet law reinstatement was "
            "associated with a 19% decrease in motorcyclist fatalities (95% CI: 9%, 28%). Conclusion: Universal "
            "helmet laws reduce motorcyclist deaths."
        )
    },
    {
        "id": "A003",
        "text": (
            "We evaluated automated speed enforcement on residential streets in Seattle, WA, 2022-2024. "
            "Using a regression-discontinuity design with 38 enforcement zones and 110 control zones, we "
            "estimated a 31% reduction in pedestrian KSI crashes (95% CI: 18%, 42%). Sensitivity analyses "
            "supported the primary finding. Limitation: Only one US city, generalizability uncertain."
        )
    },
    {
        "id": "A004",
        "text": (
            "Background: Pediatric pedestrian injuries in Mumbai, India remain a major public health concern. "
            "Methods: Cross-sectional analysis of police records, 2019-2022. Results: Children aged 5-9 "
            "accounted for 38% of pedestrian injuries, with most occurring within 500 m of schools. "
            "Conclusion: School-zone interventions are needed."
        )
    },
    {
        "id": "A005",
        "text": (
            "Background: Phoenix, AZ expanded neighborhood cooling centers and planted street trees across 24 "
            "high-heat census tracts from 2021 to 2024. Methods: Controlled before-after analysis comparing "
            "treated and matched control tracts. Results: The program was associated with a 14% reduction in "
            "heat-related emergency department visits (95% CI: 6%, 21%). Limitation: Residual confounding by "
            "neighborhood income is possible."
        )
    },
    {
        "id": "A006",
        "text": (
            "Objective: To describe the association between ambient PM2.5 and pediatric asthma emergency visits "
            "across US metropolitan areas, 2010-2019. Methods: Observational time-series analysis; no "
            "intervention. Results: Each 10 ug/m3 increase in PM2.5 was associated with a 4% increase in asthma "
            "visits (95% CI: 2%, 6%). Conclusion: Air pollution remains a major driver of pediatric asthma."
        )
    },
]

CRITERIA = (
    "Include a study only if ALL of: (1) the primary outcome is pedestrian injury/fatality/KSI OR a heat- or "
    "air-quality-related health outcome; (2) the study population is in the United States; (3) the data cover "
    "2020 or later; (4) the study EVALUATES a built-environment, urban-greening, traffic-calming, or enforcement "
    "intervention (not a purely observational association)."
)

## 3. Pattern 1 — One prompt with structured output

The simplest workflow: send the abstract and the criteria, ask the model to return JSON.

We use Pydantic to define the shape we want, then have Ollama enforce JSON output. This is the foundation for
everything that follows — agentic frameworks just chain calls like this together.

In [ ]:
from pydantic import BaseModel
from typing import Literal
import json

class ScreeningDecision(BaseModel):
    include: Literal["include", "exclude", "unclear"]
    reason: str

def screen_one(abstract_text: str) -> ScreeningDecision:
    """Run a single screening decision against the inclusion criteria."""
    response = ollama.chat(
        model=MODEL,
        format=ScreeningDecision.model_json_schema(),
        messages=[
            {"role": "system",
             "content": f"You are screening abstracts for a scoping review.\nCriteria:\n{CRITERIA}"},
            {"role": "user",
             "content": f"Abstract:\n{abstract_text}\n\nReturn a JSON decision."}
        ]
    )
    return ScreeningDecision.model_validate_json(response.message.content)

# Test on one abstract
decision = screen_one(ABSTRACTS[0]["text"])
print(decision.model_dump_json(indent=2))

Now sweep the whole list. This is already a useful little workflow — but everything is hardcoded in one function.

In [ ]:
for ab in ABSTRACTS:
    d = screen_one(ab["text"])
    print(f"{ab['id']}: {d.include.upper():8s} — {d.reason}")

## 4. Pattern 2 — Multi-agent with CrewAI

Now we make the workflow agentic. Two role-based agents collaborate:

- **Screener** decides include/exclude per the criteria.
- **Extractor** pulls structured fields from the included abstracts only.

CrewAI handles the message-passing between them. Notice the model is the same local one.

In [ ]:
from crewai import Agent, Task, Crew, LLM, Process

# Point CrewAI at your local Ollama instance.
llm = LLM(
    model=f"ollama/{MODEL}",
    base_url="http://localhost:11434",
    temperature=0.1,
)

screener = Agent(
    role="Public Health Librarian",
    goal=("Screen abstracts against the review's inclusion criteria. "
          "Return a clean list of included IDs with a one-line reason for each."),
    backstory=("You have screened thousands of abstracts for systematic reviews. "
               "You are conservative and require explicit evidence of each criterion."),
    llm=llm,
    verbose=False,
)

extractor = Agent(
    role="Urban Health Epidemiologist",
    goal=("For each included abstract, extract: study design, location, years, "
          "intervention, sample size, and the primary effect estimate with 95% CI. "
          "Format as a markdown table."),
    backstory=("You specialize in built-environment and environmental-exposure evaluations and read "
               "urban-health papers daily."),
    llm=llm,
    verbose=False,
)

# Tasks
abstract_block = "\n\n".join(f"[{a['id']}]\n{a['text']}" for a in ABSTRACTS)

screen_task = Task(
    description=(f"Inclusion criteria:\n{CRITERIA}\n\nAbstracts:\n{abstract_block}\n\n"
                 "List included IDs only, one per line, in the form: ID — short reason."),
    expected_output="A bullet list of included abstract IDs with one-line reasons.",
    agent=screener,
)

extract_task = Task(
    description=("Using the IDs the screener selected, extract fields into a markdown table. "
                 f"The full abstracts are below.\n\n{abstract_block}"),
    expected_output="A markdown table with one row per included abstract.",
    agent=extractor,
    context=[screen_task],   # extractor sees screener's output
)

crew = Crew(
    agents=[screener, extractor],
    tasks=[screen_task, extract_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
print(result.raw)

## 5. Pattern 3 — State machine with LangGraph

LangGraph expresses the same workflow as an explicit graph. You write each step as a function that takes a state
dict and returns an updated state. Edges define the flow, and you can add conditional branches.

This is more verbose than CrewAI but gives you tight control: human checkpoints, retries, and per-step inspection.

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm_lg = ChatOllama(model=MODEL, temperature=0.1, format="json")

class ReviewState(TypedDict):
    abstract_id: str
    abstract: str
    decision: str          # "include" | "exclude" | "unclear"
    reason: str
    extracted: dict        # populated only for included abstracts

def screen_node(state: ReviewState) -> ReviewState:
    """Decide whether to include this abstract."""
    msg = llm_lg.invoke([
        SystemMessage(content=f"Inclusion criteria:\n{CRITERIA}\n"
                              "Reply with JSON: {{\"decision\": \"include|exclude|unclear\", \"reason\": \"...\"}}"),
        HumanMessage(content=f"Abstract:\n{state['abstract']}")
    ])
    parsed = json.loads(msg.content)
    return {**state, "decision": parsed["decision"], "reason": parsed["reason"]}

def extract_node(state: ReviewState) -> ReviewState:
    """Extract structured fields. Only runs if include == 'include'."""
    msg = llm_lg.invoke([
        SystemMessage(content=("Extract a JSON object with keys: design, location, years, "
                               "intervention, n, effect (with 95% CI as a string), limitation. "
                               "Use null for fields not stated.")),
        HumanMessage(content=f"Abstract:\n{state['abstract']}")
    ])
    return {**state, "extracted": json.loads(msg.content)}

def route_after_screen(state: ReviewState) -> str:
    """Conditional edge: only extract if included."""
    return "extract" if state["decision"] == "include" else END

# Build the graph
g = StateGraph(ReviewState)
g.add_node("screen", screen_node)
g.add_node("extract", extract_node)
g.set_entry_point("screen")
g.add_conditional_edges("screen", route_after_screen, {"extract": "extract", END: END})
g.add_edge("extract", END)

review_app = g.compile()

# Run the graph against each abstract
for ab in ABSTRACTS:
    result = review_app.invoke({
        "abstract_id": ab["id"], "abstract": ab["text"],
        "decision": "", "reason": "", "extracted": {}
    })
    print(f"\n--- {result['abstract_id']} ---")
    print(f"Decision: {result['decision']} — {result['reason']}")
    if result.get("extracted"):
        print("Extracted:", json.dumps(result["extracted"], indent=2))

## 6. Adding a tool — agent that calls a function

The pieces above only call the LLM. Real agents call tools too: a search API, a calculator, a database.
Here we give a tiny agent a fake "PubMed search" tool and a confidence-interval calculator.

This uses LangGraph's pre-built `create_react_agent`, which wires up the tool-calling loop for you. Conceptually,
this is what **MCP** standardizes — a uniform way to expose tools to a model.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
import math

@tool
def fake_pubmed_search(query: str) -> str:
    """Search a tiny local index of urban-health abstracts. Returns matching IDs and one-line summaries."""
    hits = []
    q = query.lower()
    for ab in ABSTRACTS:
        text = ab["text"].lower()
        if any(word in text for word in q.split() if len(word) > 3):
            first_sentence = ab["text"].split(". ")[0]
            hits.append(f"{ab['id']}: {first_sentence}.")
    return "\n".join(hits) if hits else "No matches."

@tool
def wilson_ci(events: int, total: int) -> str:
    """Compute the 95% Wilson confidence interval for a proportion. Returns 'point (lower, upper)'."""
    if total == 0:
        return "undefined (n=0)"
    p = events / total
    z = 1.96
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2*total)) / denom
    half = (z * math.sqrt(p*(1-p)/total + z**2/(4*total**2))) / denom
    return f"{p:.3f} ({centre - half:.3f}, {centre + half:.3f})"

tool_agent = create_react_agent(
    model=ChatOllama(model=MODEL, temperature=0.0),
    tools=[fake_pubmed_search, wilson_ci],
)

result = tool_agent.invoke({
    "messages": [
        ("user",
         "Find studies about leading pedestrian intervals or speed enforcement, "
         "then for the first one tell me the Wilson 95% CI for 22 events out of 100.")
    ]
})

# Print the final assistant message
print(result["messages"][-1].content)

## 7. Try it yourself

Pick **one** of the following and modify the code above to do it. Each takes 10–20 minutes.

1. **Add a reviewer agent.** In the CrewAI pipeline, add a third agent ("Methodologist") whose job is to read
   the extractor's table and flag any rows where the effect size or CI looks inconsistent with the abstract.

2. **Add a citation step.** In the LangGraph version, add a node after `extract` that asks the model to write a
   one-sentence narrative summary of each included study, formatted for a paper's introduction.

3. **Replace the fake search tool.** Swap `fake_pubmed_search` for a real one — for example, hit the NCBI E-utilities
   API to query PubMed. Note: this sends your queries (not the abstracts) to NIH servers.

4. **Compare two models.** Run section 3 (single-prompt screening) with `qwen3` and again with `gemma4`.
   Where do they disagree? Which is faster?

When you write up your assignment, include:
- Your modified code.
- One screenshot of the model's output.
- A short note on what surprised you about the model's behavior, and one task you would not trust it with.

## 8. How this maps to Day 3

- **The agent loop.** Each pattern is the same loop — *perceive (read input) → decide → act → repeat* — with
  increasing structure around it. Pattern 1 is one turn; CrewAI and LangGraph add multiple turns and branching.
- **Context engineering.** The model only knows what you put in the prompt: the criteria, the abstract, and the
  role/backstory. Most of the quality comes from that context, not from the framework.
- **Tools / MCP.** Section 6 is a tool-calling agent. MCP is just a standard way to expose tools like these so
  any model can use them.
- **Guardrails.** Decisions are constrained to a fixed schema (`include|exclude|unclear`), the extractor only sees
  included abstracts, and extraction uses `null` for unstated fields.
- **Verification is the bottleneck.** Always re-read the model's include/exclude calls and extracted numbers
  against the abstracts before using them.

## 9. Notes and gotchas

- **JSON mode is not perfect.** Even with `format="json"` or a Pydantic schema, small open models occasionally
  return malformed JSON or hallucinate field values. Wrap parsing in try/except and log failures.
- **Speed scales with calls.** A 6-abstract sweep that takes under a minute becomes a long job for 60 abstracts
  and an overnight job for 600. Plan accordingly.
- **Local does not mean free of supervision.** All the warnings from the lecture still apply: hallucinated
  citations, weak math, stale knowledge. Always verify before acting on output.
- **Memory matters.** CrewAI and LangGraph both keep history in RAM. For long runs, write intermediate state to disk
  so you can resume after a crash.
- **Streaming helps perception.** For interactive demos, use `ollama.chat(..., stream=True)` and print tokens as they arrive.